In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import datetime
import os
import gc
import re
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

import logging
import socket

# =====================
# 日志配置
# =====================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("xgboost:", xgb.__version__)
import pyspark
print("pyspark:",pyspark.__version__)
print(datetime.datetime.now())

def get_local_ip():
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
    except Exception:
        ip = "127.0.0.1"
    finally:
        s.close()
    return ip

ipl = get_local_ip()

# =====================
# Spark 初始化
# =====================


os.environ['JAVA_HOME'] = '/home/jovyan/shared/jdk11'
os.environ['HADOOP_CONF_DIR'] = '/home/jovyan/shared'
os.environ['YARN_CONF_DIR'] = '/home/jovyan/shared'
os.environ['HADOOP_USER_NAME'] = 'hadoop'  # 使用系统中存在的用户


DRIVER_IP = ipl 

def create_spark_session_with_proxy(app_name="Jupyter-Spark-App", proxy_user="hadoop"):
    try:
        spark = (
            SparkSession.builder
            .appName(app_name)
            .appName("PartitionedTableWrite") 
            .master("yarn")
            .config("spark.sql.session.timeZone", "GMT-06:00")
            .config("spark.yarn.proxyUser", proxy_user)
            .config("hive.metastore.uris", "thrift://10.129.9.124:9083,thrift://10.129.9.17:9083")
            .config("spark.hadoop.fs.defaultFS", "hdfs://ns1")
            .config("spark.yarn.stagingDir", f"hdfs:///tmp/{proxy_user}/staging")
            .config("spark.executor.cores", "8")
            .config("spark.executor.memory", "16g")
            .config("spark.driver.memory", "8g")
            .config("spark.executor.instances", "16")
            .config("spark.submit.deployMode", "client")
            .config("spark.driver.host", DRIVER_IP)
            .config("spark.driver.bindAddress", "0.0.0.0")
            .config("spark.driver.port", "38007")
            .config("spark.blockManager.port", "38002")
            .config("spark.jars", "/home/jovyan/shared/shared_files/hive-jdbc-3.1.3.jar")
            # 添加组映射配置解决警告
            #.config("spark.hadoop.hadoop.security.group.mapping", "org.apache.hadoop.security.ShellBasedUnixGroupsMapping")
            #.config("spark.hadoop.hadoop.user.group.static.mapping.overrides", "admin=users,hadoop=users")
            .config("spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive", "true")
            .config("spark.hadoop.fs.log.level", "ERROR")
            .config("spark.hadoop.hive.metastore.logging.level", "ERROR")
            .enableHiveSupport()
            .getOrCreate()
        )
        logger.info("✅ SparkSession 创建成功！")
        return spark
    except Exception as e:
        logger.error(f"❌ 创建SparkSession失败: {str(e)}")
        raise

spark = create_spark_session_with_proxy()

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
#选取时间
today_str = datetime.date.today().strftime("%Y-%m-%d")

# 使用 Spark 读取数据
while True:
    try:
        df_spark_news = spark.sql("""
select apply_id,create_time,response_body from hive_idc.hello_prd.ods_mx_aiofeature_feature_cdc_request_log_df where pt='2026-03-08' AND create_time >='2026-03-01' and apply_id is not null
and apply_id in(
select apply_id from hive_idc.hello_prd.ods_mx_allinone_t_kepler_scene_metric_data_di  where scene_code='new_user_withdraw'
) order by create_time desc;
""")
        
        # 转换为 Pandas DataFrame 以复用后续处理逻辑
        df_news = df_spark_news.toPandas()
        df_news = df_news.astype(str)             
        if df_news is not None and len(df_news) > 0:
            break  # exit the loop when data exists
    except Exception as e:
        logger.error(f"Error reading data: {e}")
        pass
    
    time.sleep(300)  # wait 3 minutes before retrying

print('数据加载完成')

df_news.to_csv('/home/jovyan/shared/kimi/cdc_0308data')
print(datetime.datetime.now())